# CachePilot 真实 GPU 验收

本 notebook 只验证真实 GPU 推理，不使用 DeterministicChatBackend、fake engine 或模拟模型结果。

重要：项目和 vLLM `0.24.0` 的锁定基线是 Python `3.13.15`。vLLM `0.24.0` 的约束是 Python `<3.15`，所以 Python `3.15` 不能作为本项目的 GPU 运行环境；本 notebook 会在开头硬性拒绝不符合 `3.13.15` 的运行时。

验收范围：真实 CUDA/GPU、真实 Qwen 固定 revision、真实 TorchExecutor、真实 vLLM、真实 CachePilot Gateway、逐请求 Trace 和 Prometheus 指标。

## 1. 挂载仓库并固定工作目录

In [ ]:
from google.colab import drive
import os
import pathlib

drive.mount('/content/drive')
WORKDIR = pathlib.Path('/content/drive/MyDrive/CachePilot')
if not WORKDIR.is_dir():
    raise FileNotFoundError(f'CachePilot repository not found: {WORKDIR}')
os.chdir(WORKDIR)
print('工作目录:', WORKDIR)

## 2. GPU/Python 硬门禁

这一步失败就停止，不把 CPU、无 GPU 或错误 Python 版本的结果当成 GPU 验收。

In [ ]:
import json
import platform
import subprocess
import sys

EXPECTED_PYTHON = (3, 13, 15)
if sys.version_info[:3] != EXPECTED_PYTHON:
    raise RuntimeError(
        f'Python mismatch: expected 3.13.15, got {sys.version.split()[0]}. '
        'Python 3.15 is not supported by the locked vLLM 0.24.0 profile.'
    )
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('Real vLLM GPU validation requires Linux x86_64.')

gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True,
    text=True,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('No usable NVIDIA GPU was found by nvidia-smi.')

baseline = json.loads((WORKDIR / 'config/model.json').read_text())
dependencies = json.loads((WORKDIR / 'config/dependencies.json').read_text())
print('Python:', sys.version.replace('\n', ' '))
print('GPU:', gpu_probe.stdout.strip())
print('Model:', baseline['model_id'])
print('Model revision:', baseline['model_revision'])
print('Locked vLLM:', dependencies['profiles']['vllm_executor']['vllm'])
print('GPU_PYTHON_PREFLIGHT=PASS')

## 3. 安装锁定的真实 GPU 依赖

安装失败直接停止；不再降级为 CPU 或 fake 实现。

In [ ]:
import os
import subprocess
import sys

project_packages = [
    'fastapi==0.115.12',
    'httpx==0.28.1',
    'pydantic==2.11.4',
    'starlette==0.46.2',
    'uvicorn==0.34.2',
]
gpu_packages = [
    'torch==2.11.0',
    'transformers==5.5.3',
    'tokenizers==0.22.2',
    'safetensors==0.6.2',
    'huggingface_hub==1.5.0',
    'vllm==0.24.0',
]
def install_packages(packages):
    command = [
        sys.executable, '-m', 'pip', 'install', '--upgrade',
        '--disable-pip-version-check', '--break-system-packages', *packages,
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr[-12000:])
        raise subprocess.CalledProcessError(result.returncode, command)
install_packages(project_packages)
install_packages(gpu_packages)
os.environ['PYTHONPATH'] = str(WORKDIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
print('GPU_DEPENDENCIES_INSTALL=PASS')

In [ ]:
from importlib import metadata
import torch
import transformers

assert torch.__version__.split('+')[0] == '2.11.0', torch.__version__
assert transformers.__version__ == '5.5.3', transformers.__version__
assert metadata.version('vllm') == '0.24.0'
assert torch.cuda.is_available(), 'CUDA is not available'
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('vLLM:', metadata.version('vllm'))
print('CUDA:', torch.version.cuda)
print('Device:', torch.cuda.get_device_name(0))
print('GPU_RUNTIME=PASS')

## 4. 真实 TorchExecutor + CachePilot Gateway

加载固定 revision 的 Qwen，不注入 fake model/tokenizer。请求必须经过真实 GPU 生成，再检查 Gateway Trace 和 Prometheus。

In [ ]:
from httpx import ASGITransport, AsyncClient
from cachepilot.config.baseline import load_model_baseline
from cachepilot.executors import TorchExecutor, TorchExecutorConfig
from cachepilot.gateway.api import GatewaySettings, create_app

baseline_obj = load_model_baseline(WORKDIR / 'config/model.json')
torch_executor = TorchExecutor(
    TorchExecutorConfig(
        model_id=baseline_obj.model_id,
        tokenizer_id=baseline_obj.tokenizer_id,
        model_revision=baseline_obj.model_revision,
        tokenizer_revision=baseline_obj.tokenizer_revision,
        device='cuda',
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'float16',
        context_limit=baseline_obj.service_context_limit,
        trust_remote_code=baseline_obj.trust_remote_code,
    )
)
torch_app = create_app(
    GatewaySettings(model_id=baseline_obj.model_id),
    backend=torch_executor,
    token_counter=torch_executor,
)

async def run_real_torch_gateway():
    async with AsyncClient(
        transport=ASGITransport(app=torch_app),
        base_url='http://cachepilot',
    ) as client:
        ready = await client.get('/readyz')
        assert ready.status_code == 200 and ready.json()['status'] == 'ready'
        response = await client.post(
            '/v1/chat/completions',
            headers={'X-Tenant-ID': 'team-a', 'X-Request-ID': 'real-torch-gateway'},
            json={
                'model': baseline_obj.model_id,
                'messages': [{'role': 'user', 'content': 'Reply with a short greeting.'}],
                'stream': False,
                'max_tokens': 16,
            },
        )
        assert response.status_code == 200, response.text
        payload = response.json()
        output = payload['choices'][0]['message']['content']
        assert output.strip(), 'TorchExecutor returned empty output'
        trace_response = await client.get(
            '/v1/requests/real-torch-gateway',
            headers={'X-Tenant-ID': 'team-a'},
        )
        assert trace_response.status_code == 200
        trace = trace_response.json()['telemetry']
        assert trace['terminal_state'] == 'FINISHED'
        assert trace['prompt_tokens'] > 0
        assert trace['completion_tokens'] > 0
        assert trace['ttft_ms'] is not None
        metrics = (await client.get('/metrics')).text
        assert 'cachepilot_requests_total' in metrics
        assert 'cachepilot_ttft_seconds' in metrics
        for forbidden in ('request_id=', 'prompt=', 'prefix_key='):
            assert forbidden not in metrics
        return output, trace

torch_output, torch_trace = await run_real_torch_gateway()
print('TORCH_OUTPUT:', torch_output)
print('TORCH_TRACE:', torch_trace)
print('REAL_TORCH_GATEWAY=PASS')
print('REAL_TORCH_PROMETHEUS=PASS')

## 5. 真实 vLLM + CachePilot Gateway

vLLM 放在独立 Python 子进程中，避免 CUDA 动态库和 vLLM worker 污染 notebook 内核。该子进程会真实加载模型、生成 token、执行 abort，并通过真实 Gateway 检查 Trace 和 Prometheus。任何安装、CUDA、模型加载或请求失败都会直接失败，不降级、不 SKIP。

In [ ]:
import pathlib
import site
import subprocess
import textwrap

site_roots = {pathlib.Path(p) for p in site.getsitepackages()}
site_roots.add(pathlib.Path(site.getusersitepackages()))
library_dirs = set()
for site_root in site_roots:
    library_dirs.update(p for p in site_root.glob('nvidia/**/lib') if p.is_dir())
    torch_lib = site_root / 'torch' / 'lib'
    if torch_lib.is_dir():
        library_dirs.add(torch_lib)
library_dirs = sorted(str(p) for p in library_dirs)
smoke_env = os.environ.copy()
smoke_env['PYTHONPATH'] = str(WORKDIR) + os.pathsep + smoke_env.get('PYTHONPATH', '')
smoke_env['LD_LIBRARY_PATH'] = os.pathsep.join(
    library_dirs + ([smoke_env['LD_LIBRARY_PATH']] if smoke_env.get('LD_LIBRARY_PATH') else [])
)
smoke_env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

vllm_script = textwrap.dedent(f'''
    import asyncio
    import inspect
    import pathlib
    from importlib import metadata
    from httpx import ASGITransport, AsyncClient

    from cachepilot.config.baseline import load_model_baseline
    from cachepilot.executors import VllmExecutor, VllmExecutorConfig
    from cachepilot.gateway.api import GatewaySettings, create_app
    from cachepilot.gateway.contracts import validate_chat_completion_request

    WORKDIR = pathlib.Path({str(WORKDIR)!r})
    LOCKED_VLLM = '0.24.0'

    def request(baseline, request_id, content, max_tokens, stream=True):
        return validate_chat_completion_request(
            {{
                'model': baseline.model_id,
                'messages': [{{'role': 'user', 'content': content}}],
                'stream': stream,
                'max_tokens': max_tokens,
            }},
            {{'X-Tenant-ID': 'team-a', 'X-Request-ID': request_id}},
            configured_model=baseline.model_id,
        )

    async def main():
        assert metadata.version('vllm') == LOCKED_VLLM
        baseline = load_model_baseline(WORKDIR / 'config/model.json')
        executor = VllmExecutor(
            VllmExecutorConfig(
                model_id=baseline.model_id,
                tokenizer_id=baseline.tokenizer_id,
                model_revision=baseline.model_revision,
                tokenizer_revision=baseline.tokenizer_revision,
                dtype='float16',
                context_limit=baseline.service_context_limit,
                gpu_memory_utilization=0.75,
                trust_remote_code=baseline.trust_remote_code,
                expected_vllm_version=LOCKED_VLLM,
            )
        )
        try:
            direct_request = request(baseline, 'real-vllm-direct', 'Reply with a short greeting.', 8)
            direct_items = [item async for item in executor.generate(direct_request)]
            direct_output = ''.join(item.text for item in direct_items)
            usage = executor.usage('real-vllm-direct')
            assert direct_output.strip()
            assert usage is not None
            assert usage.completion_tokens == sum(item.token_count for item in direct_items)
            assert usage.total_tokens == usage.prompt_tokens + usage.completion_tokens

            app = create_app(
                GatewaySettings(model_id=baseline.model_id),
                backend=executor,
                token_counter=executor,
            )
            async with AsyncClient(
                transport=ASGITransport(app=app),
                base_url='http://cachepilot',
            ) as client:
                response = await client.post(
                    '/v1/chat/completions',
                    headers={{'X-Tenant-ID': 'team-a', 'X-Request-ID': 'real-vllm-gateway'}},
                    json={{
                        'model': baseline.model_id,
                        'messages': [{{'role': 'user', 'content': 'Reply with a short greeting.'}}],
                        'stream': False,
                        'max_tokens': 8,
                    }},
                )
                assert response.status_code == 200, response.text
                assert response.json()['choices'][0]['message']['content'].strip()
                trace_response = await client.get(
                    '/v1/requests/real-vllm-gateway',
                    headers={{'X-Tenant-ID': 'team-a'}},
                )
                assert trace_response.status_code == 200
                trace = trace_response.json()['telemetry']
                assert trace['terminal_state'] == 'FINISHED'
                assert trace['prompt_tokens'] > 0
                assert trace['completion_tokens'] > 0
                assert trace['ttft_ms'] is not None
                metrics = (await client.get('/metrics')).text
                assert 'cachepilot_requests_total' in metrics
                assert 'cachepilot_ttft_seconds' in metrics
                for forbidden in ('request_id=', 'prompt=', 'prefix_key='):
                    assert forbidden not in metrics
                print('VLLM_GATEWAY_TELEMETRY=PASS')
                print('VLLM_PROMETHEUS_LOW_CARDINALITY=PASS')

            cancel_request = request(baseline, 'real-vllm-cancel', 'Write a long numbered list.', 64)
            stream = executor.generate(cancel_request)
            first_delta = await anext(stream)
            await executor.cancel('real-vllm-cancel', reason='explicit')
            remaining = [item async for item in stream]
            assert first_delta.text
            assert not remaining
            assert executor.cancel_reasons['real-vllm-cancel'] == 'explicit'
            print('VLLM_OUTPUT:', direct_output)
            print('VLLM_USAGE:', usage)
            print('VLLM_ABORT=PASS')
            print('VLLM_REAL_GPU=PASS')
        finally:
            shutdown = getattr(executor.engine, 'shutdown_background_loop', None)
            if callable(shutdown):
                result = shutdown()
                if inspect.isawaitable(result):
                    await result

    asyncio.run(main())
''')

vllm_run = subprocess.run(
    [sys.executable, '-c', vllm_script],
    cwd=WORKDIR,
    env=smoke_env,
    text=True,
    capture_output=True,
    timeout=1800,
)
print(vllm_run.stdout[-12000:])
if vllm_run.returncode != 0:
    print(vllm_run.stderr[-12000:])
    raise RuntimeError('Real vLLM GPU validation failed.')
assert 'VLLM_REAL_GPU=PASS' in vllm_run.stdout
assert 'VLLM_GATEWAY_TELEMETRY=PASS' in vllm_run.stdout
print('REAL_VLLM_GATEWAY=PASS')

## 6. 最终结论

只有以下结果全部出现，才算真实 GPU 验收通过：

- `GPU_PYTHON_PREFLIGHT=PASS`
- `GPU_RUNTIME=PASS`
- `REAL_TORCH_GATEWAY=PASS`
- `REAL_VLLM_GATEWAY=PASS`
- `VLLM_ABORT=PASS`
- `VLLM_GATEWAY_TELEMETRY=PASS`
- `VLLM_PROMETHEUS_LOW_CARDINALITY=PASS`

这些结果证明真实模型生成、Gateway 接入、取消、逐请求 Trace 和 Prometheus 指标都在当前 GPU 环境执行。它们仍不等于物理 Prefix 命中已可观测；如果执行器没有提供可验证的物理命中信号，该结论仍应显示为“不可观测”。

In [ ]:
assert torch_output.strip()
assert vllm_run.returncode == 0
assert 'VLLM_REAL_GPU=PASS' in vllm_run.stdout
assert 'VLLM_GATEWAY_TELEMETRY=PASS' in vllm_run.stdout
print('REAL_GPU_ACCEPTANCE=PASS')
print('No deterministic backend or fake engine was used by this notebook.')